In [ ]:
%pip install rustbpe
%pip install datasets
%pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.3 MB/s eta 0:00:00


In [ ]:
# Imports (Removed Google Drive Mount)
import os
import math

import numpy as np
import wandb
from datasets import load_dataset

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import json
import tiktoken
from textwrap import fill
import rustbpe
from rustbpe import Tokenizer
from dataclasses import dataclass


Mounted at /content/drive
Using device: cuda


### Debug cuda

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

### Import our Data Set

In [ ]:
# use name="sample-10BT" to use the 10BT sample
fw = load_dataset("HuggingFaceFW/fineweb", name="CC-MAIN-2024-10", split="train", streaming=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_7871/1978799342.py", line 2, in <cell line: 0>
    fw = load_dataset("HuggingFaceFW/fineweb", name="CC-MAIN-2024-10", split="train", streaming=True)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1392, in load_dataset
    builder_instance = load_dataset_builder(
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1004, in dataset_module_factory
    ).get_module()
      ^^^^^^^^^^^^
  File "/usr/


KeyboardInterrupt



### Create our text

In [ ]:
from datasets import load_dataset

# Updated to a relative local path. Adjust this to where your dataset is stored locally.
local_path = "./datasets/fineweb_sample"
fw = load_dataset("parquet", data_files=f"{local_path}/sample/10BT/*.parquet", split="train", streaming=True)


In [ ]:
def batch_iterator(fw, batch_size=100000):
    batch = []
    for i, a in enumerate(fw):
        batch.append("<|startofseq|>" + a["text"])
        if len(batch) == batch_size:
            yield batch
            batch = []

SPECIAL_TOKENS = ["<|startofseq|>"]

VOCAB_SIZE = 32768
SEQ_LEN = 1024+1
BATCH_SIZE = 8

# Create tokenizer and train on your data
batch_itr = batch_iterator(fw)

training_texts = next(batch_itr)


### Create Tokenizer (Only run Once)

In [ ]:
# creates a tokenzier (run only once)
from textwrap import fill
import rustbpe
from rustbpe import Tokenizer

# creates custom tokenizer from our data
tokenizer = Tokenizer()
tokenizer.train_from_iterator(
    training_texts,
    vocab_size=VOCAB_SIZE,
)

# Save Tokenizer (Updated path)
import json

data = {
    "pattern": tokenizer.get_pattern(),
    "mergeable_ranks": {k.hex(): v for k, v in tokenizer.get_mergeable_ranks()}
}
with open("./tokenizer_meta.json", "w") as f:
    json.dump(data, f)


# Check vocabulary size
print("Vocab: ", tokenizer.vocab_size)


### Load our Tokenizer and Give Embeddings

In [ ]:
# load our created tokenizer

# 1. Open and read the saved JSON file (Updated path)
with open("./tokenizer_meta.json", "r") as f:
    saved_data = json.load(f)

# 2. Reverse the hex conversion to get your raw bytes back
loaded_ranks = {bytes.fromhex(k): v for k, v in saved_data["mergeable_ranks"].items()}
loaded_pattern = saved_data["pattern"]

# 3. Load it into TikToken (The Inference Engine)
# We also explicitly register your special token here!
tokenizer = tiktoken.Encoding(
    name="nanobot_tokenizer",
    pat_str=loaded_pattern,
    mergeable_ranks=loaded_ranks,
    special_tokens={"<|startofseq|>": 32767}
)

# 4. Verify it loaded correctly
print("Loaded Vocab Size: ", tokenizer.n_vocab)
# Verify it loaded correctly


# Batch encode (parallel)
all_ids = tokenizer.encode_batch(
    training_texts,
    allowed_special="all"
)
print("Total Tokens: ", sum(len(x) for x in all_ids))



# Create our sequences
sequences = []
sequence = []
for tokens in all_ids:
    # adds all tokens
    sequence.extend(tokens)
    # in each step keeps up to SEQ_LEN tokens in each sequence,
    # and moves the remaining tokens into next sequence
    while len(sequence) >= SEQ_LEN:
        sequences.append(sequence[:SEQ_LEN])
        sequence = sequence[SEQ_LEN:]

print("Sequences: ", len(sequences))
print("Total Tokens (Post batching): ", len(sequences) * SEQ_LEN)
print(tokenizer.decode(sequences[0]))

# creates batches sequences of BATCH_SZIE
batches = []
batch = []
for s in sequences:
    batch.append(s)
    if len(batch) == BATCH_SIZE:
        batches.append(batch)
        batch = []


Loaded Vocab Size:  32768
Total Tokens:  69690669
Sequences:  67990
Total Tokens (Post batching):  69689750
 exclusions|Viewing Single Post From: Spoilers for the Week of February 11th|
|Lil||Feb 1 2013, 09:58 AM|
Don't care about Chloe/Taniel/Jen-Jen. Don't care about Sami, really, but hoping that we get some good "SAMANTHA GENE!!" Marlena Death-Stares out of it. And "newfound" feelings. Please. If only.
STEFANO!! STEFANO, STEFANO, STEFANO!!!! :cheer:
|Spoilers for the Week of February 11th · DAYS: News, Spoilers & Discussion| exclusions*sigh* Fundamentalist community, let me pass on some advice to you I learned from the atheistic community:
If you have set yourself on fire, do not run.
Okay? Okay?? Please?
Look, D, you had two months to say to Harvard in private emails, "Im sorry, I shouldnt have been using that animation in my paid presentations. I wont use it again. I really do like 'Inner Life', though, and would love to use it in classroom presentations, from the BioVisions site,

### Custom Tokenizer Implementation (Not needed)

In [ ]:
# @title
# Dont use
# Used to give conceptual idea of a tokenizer

from textwrap import fill
import rustbpe
from rustbpe import Tokenizer

class HFCompatibleTokenizer:
    def __init__(self, rust_tokenizer, max_length=SEQ_LEN, pad_token_id=0):
        self.tokenizer = rust_tokenizer
        self.max_length = max_length
        self.pad_token_id = pad_token_id

    def __call__(self, texts, padding="max_length", truncation=True, max_length=None):
        if max_length is None:
            max_length = self.max_length

        # Ensure texts is a list for batch processing
        if isinstance(texts, str):
            texts = [texts]

        # Use your custom batch_encode
        all_ids = self.tokenizer.batch_encode(texts)

        input_ids = []
        attention_masks = []

        for ids in all_ids:
            # Handle Truncation
            if truncation and len(ids) > max_length:
                ids = ids[:max_length]

            # Create attention mask (1 for real tokens)
            mask = [1] * len(ids)

            # Handle Padding
            if padding == "max_length":
                pad_len = max_length - len(ids)
                if pad_len > 0:
                    ids = ids + [self.pad_token_id] * pad_len
                    mask = mask + [0] * pad_len

            input_ids.append(ids)
            attention_masks.append(mask)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_masks
        }

In [ ]:
batches[0]

[[32768,
  4436,
  348,
  314,
  425,
  2283,
  981,
  578,
  354,
  9854,
  1666,
  2282,
  3022,
  282,
  640,
  300,
  4469,
  716,
  578,
  16265,
  339,
  263,
  4030,
  496,
  2127,
  282,
  1594,
  16207,
  46,
  2500,
  13249,
  32,
  7931,
  37,
  1666,
  426,
  4405,
  559,
  1641,
  413,
  3038,
  2850,
  10414,
  2748,
  456,
  26220,
  1337,
  343,
  32,
  6227,
  37,
  288,
  14052,
  797,
  285,
  32,
  6183,
  37,
  510,
  4174,
  14485,
  723,
  41,
  1375,
  257,
  13928,
  45,
  6499,
  288,
  627,
  10414,
  2748,
  354,
  425,
  2293,
  687,
  14485,
  723,
  356,
  14052,
  797,
  561,
  300,
  943,
  1317,
  282,
  1639,
  1956,
  801,
  5131,
  944,
  46,
  3962,
  263,
  728,
  3587,
  288,
  560,
  402,
  559,
  1381,
  23699,
  3698,
  282,
  1375,
  339,
  428,
  616,
  9035,
  6850,
  1202,
  263,
  1717,
  288,
  263,
  1330,
  21135,
  449,
  257,
  880,
  2057,
  8686,
  1846,
  308,
  71,
  4318,
  14866,
  1388,
  59,
  7947,
  288,
  263,
  1330,
  77

### Mode Architecture

In [ ]:
#%%writefile attn.py
@dataclass
class GPTConfig():
    layers: int = 16
    d_embed: int = 1024
    d_model: int = 1024 # 128 d_model, 8 heads,
    num_heads: int = 8
    d_head: int = d_model // num_heads
    seq_len: int = SEQ_LEN
    vocab: int = VOCAB_SIZE

In [ ]:
def norm(x):
    return F.rms_norm(x, (x.size(-1), ))

class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.up = nn.Linear(config.d_embed, config.d_embed*4, bias = False)
        self.down = nn.Linear(config.d_embed*4, config.d_embed, bias = False)

        # to prevent exploding variance, we want to scale down by sqrt(2*Layers*d_embed)
        factor = (2 * config.layers) ** 0.5
        nn.init.normal_(self.down.weight, 0, 1 / (config.d_embed ** 0.5 * factor))

    def forward(self, x):
        return self.down(F.relu(self.up(x)).square())

def apply_rotary(x, d, cos_sin, start_pos = 0):
    cos, sin = cos_sin

    # Slice cos and sin up to the current sequence length (x.shape[1])
    seq_len = x.shape[1]
    cos = cos[:, start_pos : start_pos + seq_len, :, :]
    sin = sin[:, start_pos : start_pos + seq_len, :, :]

    # x1 gets the first half of the features, y1 gets the second half
    x1, y1 = x[...,:d//2], x[...,d//2:]
    x2 = x1 * cos - y1 * sin
    y2 = x1 * sin + y1 * cos
    return torch.cat([x2, y2], dim=-1)


class Attention(nn.Module):
    def __init__(self, config: GPTConfig):
        '''
        Creates attention block
        Defines wq, wk, wv to be d_model x d_embed linear transformation with random values
        Defines w0 to be d_embed x d_model linear transformation with random values
        '''
        super().__init__()
        self.config = config
        self.wq = nn.Linear(config.d_embed, config.d_model, bias = False)
        self.wk = nn.Linear(config.d_embed, config.d_model, bias = False)
        self.wv = nn.Linear(config.d_embed, config.d_model, bias = False)
        self.w0 = nn.Linear(config.d_model, config.d_embed, bias = False)

        # Mask
        self.register_buffer("bias", torch.tril(torch.ones(config.seq_len, config.seq_len))
                                     .view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x, cos_sin, start_pos=0, kv_cache=None):
        '''
        One attention block pass
        '''
        # Creates tensor q, k, v to be Batchsize x SeqLen x NumHeads x NumHeadValues
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.config.num_heads, self.config.d_head)
        k = self.wk(x).view(B, T, self.config.num_heads, self.config.d_head)
        v = self.wv(x).view(B, T, self.config.num_heads, self.config.d_head)


        # Applies softmaxed attention information
        q = q.transpose(1, 2) # (B, num heads, T, d_head)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        q, k = norm(q), norm(k)

         # Adds positional information to q and k
        q = apply_rotary(q, self.config.d_head, cos_sin, start_pos)
        k = apply_rotary(k, self.config.d_head, cos_sin, start_pos)

        # kv_cahce injection
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            # Concatenate the old past memory with our new current memory along the Sequence dimension (dim=2)
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)


        # Save the updated memory to pass back out
        # .detach() prevents memory leaks during inference
        new_kv_cache = (k.detach(), v.detach())

        # Attention Calculation
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))

        # We only need the causal mask if T > 1 (e.g., during the initial prompt processing)
        # If T == 1 (generating one word at a time), Q only looks back, no mask needed!
        if T > 1:
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))

        # print(att)
        att = F.softmax(att, dim=-1)
        # print(att)
        att = att @ v

        # Retuns data back to original shape
        att = att.transpose(1, 2)
        att = att.contiguous()
        att = att.view(B, T, C)
        att = self.w0(att)
        return att, new_kv_cache

class Block(nn.Module):
    """
    Attention + MLP block
    """
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.mlp = MLP(config = config)
        self.attention = Attention(config = config)

    def forward(self, x, cos_sin, start_pos=0, kv_cache=None):
        att_out, new_kv_cache = self.attention(norm(x), cos_sin, start_pos, kv_cache)
        x = x + att_out
        x = x + self.mlp(norm(x))
        return x, new_kv_cache

class GPT(nn.Module):

    def init_rope(self, seq_len, d, base = 10000):
        """
        Creates positional cos and sin encodings for each
        """
        # Multiplies each value by their custom theta value
        values = torch.arange(0, seq_len, dtype=torch.float32, device=device)
        theta = 1.0 / (base ** (2 * (torch.arange(0, d // 2, dtype=torch.float32, device=device)) / d))
        pos_enc = torch.outer(values, theta)
        # Maps as cosine and sine embeddings
        cos, sin = torch.cos(pos_enc).to(device=device), torch.sin(pos_enc).to(device=device),
        cos, sin = cos[None, :, None, :], sin[None, :, None, :]
        return cos, sin

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        # Embeddings for all words in vocab
        self.wte = nn.Embedding(config.vocab, config.d_embed)

        self.blocks = nn.ModuleList([Block(self.config) for i in range(self.config.layers)])
        self.ln_f = nn.LayerNorm(self.config.d_embed)
        self.head = nn.Linear(self.config.d_embed, self.config.vocab, bias = False)
        self.head.weight = self.wte.weight
        nn.init.normal_(self.wte.weight, std=1/config.d_embed**0.5)

        self.cos, self.sin = self.init_rope(self.config.seq_len, self.config.d_head)
        #Register buffers to handle CPU/GPU movement
        # self.register_buffer("cos", self.cos)
        # self.register_buffer("sin", self.sin)

    def compute_param(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


    def forward(self, x, target = None, start_pos=0, past_key_values=None):
        x = norm(self.wte(x))
        cos_sin = (self.cos, self.sin)

        new_past_key_values = []

        # Pipe the correct cache to the correct layer block and append new block
        for i, block in enumerate(self.blocks):
            layer_cache = past_key_values[i] if past_key_values is not None else None
            x, new_layer_cache = block(x, cos_sin, start_pos, layer_cache)
            new_past_key_values.append(new_layer_cache)

        x = self.ln_f(x)
        x = self.head(x)

        if target is not None:
            loss = F.cross_entropy(x.reshape(-1, x.size(-1)), target.reshape(-1), ignore_index=-1)
            return loss
        else:
            return x, new_past_key_values



### Train our Model

In [ ]:
def lr_schedule(step):
    if step < 800:
        return step / 40
    if step < 1200:
        return (1200-step-1) / 400 * 20 + 1
    elif step < STEPS * 0.9:
        return 1.0
    else:
        return (STEPS - step) / (STEPS * 0.1)

In [ ]:
# @title
from huggingface_hub import snapshot_download

# make this only download the sample-10BT
local_path = snapshot_download(
    "HuggingFaceFW/fineweb",
    repo_type="dataset",
    allow_patterns=["sample/10BT/*"],
    max_workers=1
)
print(f"Downloaded to: {local_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--HuggingFaceFW--fineweb/snapshots/9bb295ddab0e05d785b879661af7260fed5140fc


In [ ]:
import json, tiktoken
import numpy as np
import torch

# Updated path
with open("./tokenizer_meta.json") as f:
    data = json.load(f)
tokenizer = tiktoken.Encoding(
    name="tokenizer.tok",
    pat_str=data["pattern"],
    mergeable_ranks={bytes.fromhex(k): v for k, v in data["mergeable_ranks"].items()},
    special_tokens={},
)

def token_iter(docs, tokenizer, batch_size=128):
    SEQ_LEN = 1024+1
    batch_tokens = []
    documents = []
    for a in docs:  # Use already-loaded docs
        documents.append("<|startofseq|>" + a["text"])
        if len(documents) >= 128:
            for tokens in tokenizer.encode_batch(documents):
                batch_tokens.extend(tokens)
            documents = []

            while len(batch_tokens) >= batch_size * SEQ_LEN:
                yield torch.tensor(batch_tokens[:batch_size * SEQ_LEN], dtype=torch.int32).reshape((batch_size, SEQ_LEN))
                batch_tokens = batch_tokens[batch_size * SEQ_LEN:]


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import wandb
import os
import time
USE_WANDB=True

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

BATCH_SIZE = 26
GRAD_ACCUM = 1
STEPS = 32_000
LR = 3e-4
SEQ_LEN = 1024

model = GPT(GPTConfig())
# model.compile()
model.compile(mode = "max-autotune-no-cudagraphs")
dev="cuda"
model.to(dev)
# print(model)
print(sum(p.numel() for p in model.parameters() if p.requires_grad))


opt = AdamW(model.parameters(), lr=LR, betas=(0.9, 0.99), fused=True)
train_iter = token_iter(fw, tokenizer, BATCH_SIZE)


scheduler = LambdaLR(opt, lr_schedule)
losses = []
loss_acc=0
if USE_WANDB:
    # Updated to use standard environment variables instead of userdata
    wandb.login(key=os.getenv('WANDB_API_KEY'))
    run = wandb.init(entity="raymondsh0705-the-university-of-texas-at-austin", project="nanobot-1", config=dict(
        LR=LR, GRAD_ACCUM=GRAD_ACCUM, STEPS=STEPS, BATCH_SIZE=BATCH_SIZE, SEQ_LEN=SEQ_LEN)
)


### -- STORE AND LOAD MODEL --
import os

# Updated checkpoint path for local repo
checkpoint_path = './nanobot_pro_checkpoint.pt'

start_step = 0

# Check for existing memory
if os.path.exists(checkpoint_path):
    print("Found checkpoint! Restoring nanobot's memory...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_step = checkpoint['step'] + 1 # Start on the next step
    print(f"Resuming exactly from Step {start_step}")
else:
    print("No checkpoint found. Starting fresh!")





last_time = time.time()

# Update the range to start at 'start_step'
for i in range(start_step, STEPS):
    batch = next(train_iter)
    batch = batch.pin_memory().to(dev, non_blocking=True)

    input = batch[:, :-1]
    target = batch[:, 1:]

    torch.compiler.cudagraph_mark_step_begin()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        loss = model(input, target) / GRAD_ACCUM
    loss.backward()
    loss_acc += loss.item()

    if (i+1) % GRAD_ACCUM == 0:
        tps = BATCH_SIZE * SEQ_LEN * GRAD_ACCUM / (time.time()-last_time)

        if (i+1) % 50 == 0:
            print(f"step {i+1} loss {loss_acc:.4f} lr {opt.param_groups[0]['lr']:.7f}")

        if (i+1) % 2 == 0:
            if USE_WANDB:
                run.log({"loss": loss_acc, "lr": opt.param_groups[0]['lr'], "tps": tps}, step=i+1)

        opt.step()
        opt.zero_grad(set_to_none=True)
        losses.append(loss_acc)
        loss_acc = 0
        last_time = time.time()

    scheduler.step()

    # Save a checkpoint every 500 steps
    if (i+1) % 500 == 0:
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'step': i
        }
        torch.save(checkpoint, checkpoint_path)
        print(f"--> Nanobot memory saved at Step {i+1} <--")

# Final save when the loop finishes completely
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'step': STEPS
}
torch.save(checkpoint, checkpoint_path)
print("Training Complete. Final checkpoint saved!")


NameError: name 'GPT' is not defined

### Chatbot Supervised Fine Tuning
- What is Block Diagonal Attention?

In [ ]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/ultrachat_200k")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

DatasetDict({
    train_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 207865
    })
    test_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 23110
    })
    train_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 256032
    })
    test_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 28304
    })
})


##### Resize our model (Run once)

In [ ]:
import torch
import torch.nn as nn

# Initialize the model with the OLD config so shapes match your saved file
config = GPTConfig()
config.vocab = 32768
model = GPT(config)

# load the raw checkpoint file into memory (Updated path)
checkpoint = torch.load('./nanobot_pro_checkpoint.pt')
state_dict = checkpoint['model_state_dict']

# fix naming mismatch
if "word_embeddings.weight" in state_dict:
    print("Renaming 'word_embeddings.weight' to 'wte.weight' to match architecture...")
    state_dict["wte.weight"] = state_dict.pop("word_embeddings.weight")
if "lm_head.weight" in state_dict and "head.weight" not in state_dict:
    state_dict["head.weight"] = state_dict.pop("lm_head.weight")

# load the corrected dictionary into the model!
model.load_state_dict(state_dict, strict=False)

# Define the new vocabulary sizes
NEW_VOCAB = 32772
OLD_VOCAB = 32768

print("Performing brain surgery to add new tokens...")

# --- SURGERY ON THE EMBEDDINGS (model.wte) ---
old_wte_data = model.wte.weight.data
embed_dim = old_wte_data.shape[1]

# Create a new, larger embedding layer
new_wte = nn.Embedding(NEW_VOCAB, embed_dim)

# Copy the old knowledge into the top part of the new layer
new_wte.weight.data[:OLD_VOCAB, :] = old_wte_data

# Replace the embedding layer in the model
model.wte = new_wte

# --- SURGERY ON THE OUTPUT LAYER (model.head) ---
# Create a new, larger linear layer
new_head = nn.Linear(embed_dim, NEW_VOCAB, bias=False)

# Re-tie the weights!
new_head.weight = model.wte.weight
model.head = new_head

# Update the config to reflect reality
model.config.vocab = NEW_VOCAB

print("Surgery complete! New vocab size:", model.config.vocab)

model.to("cuda")
model.compile(mode="max-autotune-no-cudagraphs")


Renaming 'word_embeddings.weight' to 'wte.weight' to match architecture...
Performing brain surgery to add new tokens...
Surgery complete! New vocab size: 32772


#### Hello

Load tokenizer with updated vocab size

In [ ]:
# Load tokenizer for new SFT (Updated path)
with open("./tokenizer_meta.json", "r") as f:
    saved_data = json.load(f)

loaded_ranks = {bytes.fromhex(k): v for k, v in saved_data["mergeable_ranks"].items()}
loaded_pattern = saved_data["pattern"]

# ADD THE NEW SFT TOKENS
tokenizer = tiktoken.Encoding(
    name="nanobot_tokenizer",
    pat_str=loaded_pattern,
    mergeable_ranks=loaded_ranks,
    special_tokens={
        "<|startofseq|>": 32767,
        "<|endoftext|>": 32768,
        "<|user|>": 32769,
        "<|assistant|>": 32770,
        "<|pad|>": 32771
    }
)

print("Loaded Vocab Size: ", tokenizer.n_vocab)
# This should now print 32772!


Loaded Vocab Size:  32772


In [ ]:
### recreate our text inputs
SEQ_START = "<|startofseq|>"
USER = "<|user|>"
ASSISTANT = "<|assistant|>"
SEQ_END = "<|endoftext|>"
try:
    PAD_TOKEN_ID = tokenizer.encode("<|pad|>", allowed_special="all")[0]
except:
    PAD_TOKEN_ID = 0

VOCAB_SIZE = 32772
SEQ_LEN = 1024 + 1
BATCH_SIZE = 8

def batch_iterator_chatbot(ds, batch_size = BATCH_SIZE, seq_len = SEQ_LEN):
  TARGET_LEN = seq_len
  batch = []


  for i, row in enumerate(ds):
    string = f"{SEQ_START}\n"

    for message in row["messages"]:
        if message["role"] == "user":
            string += f"{USER}\n{message['content']}\n"
        elif message["role"] == "assistant":
            string += f"{ASSISTANT}\n{message['content']}\n"

    string += f"{SEQ_END}"
    batch.append(string)


    # reached our batch size limit
    if len(batch) == batch_size:
      token_lists = tokenizer.encode_batch(batch, allowed_special="all")

      final_tensor_grid = []

      # Loop through the ragged lists and pad/truncate them to fit seq_len
      for tokens in token_lists:
          if len(tokens) > TARGET_LEN:
              tokens = tokens[:TARGET_LEN]
          else:
              padding_needed = TARGET_LEN - len(tokens)
              tokens = tokens + [PAD_TOKEN_ID] * padding_needed

          final_tensor_grid.append(tokens)


      # Yield the perfectly rectangular batch to the GPU
      yield torch.tensor(final_tensor_grid, dtype=torch.long)

      # Reset the string batch for the next round
      batch = []


Training Loop

In [ ]:
def lr_schedule(step):
    if step < 800:
        return step / 40
    if step < 1200:
        return (1200-step-1) / 400 * 20 + 1
    elif step < STEPS * 0.9:
        return 1.0
    else:
        return (STEPS - step) / (STEPS * 0.1)

In [ ]:
import os
import torch
import torch.optim as optim
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import wandb
import time
USE_WANDB=True

BATCH_SIZE = 26
GRAD_ACCUM = 1
STEPS = 32_000
LR = 3e-4
SEQ_LEN = 1024

# Updated for local saving
save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

# enable TF32 for speed
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# Model & Optimizer Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

# (Make sure LR, BATCH_SIZE, SEQ_LEN, GRAD_ACCUM, and STEPS are defined in your notebook!)
optimizer = optim.AdamW(model.parameters(), lr=LR)

# Scheduler & WandB Setup
# Note: Ensure your `lr_schedule` function is defined before running this cell!
scheduler = LambdaLR(optimizer, lr_schedule)
losses = []
loss_acc = 0
USE_WANDB = True # Toggle this to False if you ever want to test without logging

if USE_WANDB:
    wandb.login(key=os.getenv('WANDB_API_KEY'))
    run = wandb.init(
        entity="raymondsh0705-the-university-of-texas-at-austin",
        project="nanobot-1",
        config=dict(LR=LR, GRAD_ACCUM=GRAD_ACCUM, STEPS=STEPS, BATCH_SIZE=BATCH_SIZE, SEQ_LEN=SEQ_LEN)
    )

# Grab Special Tokens
USER_TOKEN = tokenizer.encode("<|user|>", allowed_special="all")[0]
ASSISTANT_TOKEN = tokenizer.encode("<|assistant|>", allowed_special="all")[0]

try:
    PAD_TOKEN = tokenizer.encode("<|pad|>", allowed_special="all")[0]
except:
    PAD_TOKEN = 0

NUM_EPOCHS = 3
train_ds = ds["train_sft"]
SAVE_EVERY_N_STEPS = 500

# THE MAIN TRAINING LOOP
for epoch in range(NUM_EPOCHS):
    print(f"\n--- Starting Epoch {epoch+1} ---")

    for step, batch in enumerate(batch_iterator_chatbot(train_ds)):

        # slice on the CPU FIRST
        x = batch[:, :-1]
        y = batch[:, 1:].clone()

        # APPLY LOSS MASKING (On the CPU)
        y[y == PAD_TOKEN] = -1

        for row in range(y.size(0)):
            in_user_prompt = True

            for col in range(y.size(1)):
                token = y[row, col].item()

                if token == ASSISTANT_TOKEN:
                    in_user_prompt = False
                elif token == USER_TOKEN:
                    in_user_prompt = True

                if in_user_prompt:
                    y[row, col] = -1

        # Move processed tensors to GPU
        x, y = x.to(device), y.to(device)

        # Forward Pass
        loss = model(x, target=y)
        loss_acc += loss.item() # Accumulate loss for tracking

        # Backward Pass & Optimize
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step() # Step the scheduler after the optimizer

        # Logging
        if step % 10 == 0:
            current_lr = scheduler.get_last_lr()[0]
            avg_loss = loss_acc / 10 if step > 0 else loss_acc

            print(f"Epoch {epoch+1} | Step {step} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

            if USE_WANDB:
                wandb.log({
                    "train/loss": avg_loss,
                    "train/learning_rate": current_lr,
                    "epoch": epoch + 1,
                    "step": step
                })

            losses.append(avg_loss)
            loss_acc = 0 # Reset accumulator after logging

        # PERIODIC CHECKPOINTING (Overwrites the same file!)
        if step > 0 and step % SAVE_EVERY_N_STEPS == 0:
            # Notice the static file name here
            checkpoint_path = os.path.join(save_dir, "nanobot_pro_checkpoint.pt")
            torch.save({
                'epoch': epoch,
                'step': step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': loss.item(),
            }, checkpoint_path)
            print(f"--> Overwrote periodic checkpoint: {checkpoint_path}")

    # END OF EPOCH CHECKPOINTING (Overwrites the same file!)
    # Notice the static file name here as well
    epoch_checkpoint_path = os.path.join(save_dir, "nanobot_pro_checkpoint.pt")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss.item(),
    }, epoch_checkpoint_path)
    print(f"*** Overwrote END OF EPOCH checkpoint: {epoch_checkpoint_path} ***")

# Clean up WandB connection when totally done
if USE_WANDB:
    wandb.finish()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/learning_rate,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/loss,█▇▆▄▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▂▂▂▂▂▂▁▁▂
epoch,1
step,380
train/learning_rate,0.00286
train/loss,1.82661



--- Starting Epoch 1 ---
Epoch 1 | Step 0 | Loss: 2.2969 | LR: 0.000007
Epoch 1 | Step 10 | Loss: 2.3731 | LR: 0.000082
Epoch 1 | Step 20 | Loss: 2.2672 | LR: 0.000157
Epoch 1 | Step 30 | Loss: 2.2556 | LR: 0.000232
Epoch 1 | Step 40 | Loss: 2.0582 | LR: 0.000307
Epoch 1 | Step 50 | Loss: 2.0352 | LR: 0.000382
Epoch 1 | Step 60 | Loss: 2.0685 | LR: 0.000457
Epoch 1 | Step 70 | Loss: 1.7875 | LR: 0.000532
Epoch 1 | Step 80 | Loss: 1.8938 | LR: 0.000607
Epoch 1 | Step 90 | Loss: 1.8510 | LR: 0.000682
Epoch 1 | Step 100 | Loss: 1.8246 | LR: 0.000757
Epoch 1 | Step 110 | Loss: 1.8653 | LR: 0.000832
Epoch 1 | Step 120 | Loss: 1.6904 | LR: 0.000907
Epoch 1 | Step 130 | Loss: 1.7664 | LR: 0.000982
Epoch 1 | Step 140 | Loss: 1.6931 | LR: 0.001057
Epoch 1 | Step 150 | Loss: 1.5029 | LR: 0.001132
Epoch 1 | Step 160 | Loss: 1.5442 | LR: 0.001208
Epoch 1 | Step 170 | Loss: 1.5357 | LR: 0.001283
Epoch 1 | Step 180 | Loss: 1.5141 | LR: 0.001357
Epoch 1 | Step 190 | Loss: 1.3908 | LR: 0.001432
Epoch

KeyboardInterrupt: 

In [ ]:
# Removed Google Drive Mount

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Use Model

In [ ]:
# IMPORT OUR MODEL
import torch
import os

model = GPT(GPTConfig())

# Updated checkpoint path
checkpoint_path = './nanobot_pro_checkpoint.pt'


if os.path.exists(checkpoint_path):
    print("Found checkpoint! Restoring nanobot's memory...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])

# put the model in evaluation mode (turns off dropout and other training quirks)
model.to("cuda")
model.eval()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found checkpoint! Restoring nanobot's memory...


GPT(
  (wte): Embedding(32772, 1024)
  (blocks): ModuleList(
    (0-15): 16 x Block(
      (mlp): MLP(
        (up): Linear(in_features=1024, out_features=4096, bias=False)
        (down): Linear(in_features=4096, out_features=1024, bias=False)
      )
      (attention): Attention(
        (wq): Linear(in_features=1024, out_features=1024, bias=False)
        (wk): Linear(in_features=1024, out_features=1024, bias=False)
        (wv): Linear(in_features=1024, out_features=1024, bias=False)
        (w0): Linear(in_features=1024, out_features=1024, bias=False)
      )
    )
  )
  (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=1024, out_features=32772, bias=False)
)

In [ ]:
def generate_response(prompt_text, max_new_tokens=150, device="cuda"):
    # format prompt to model
    formatted_prompt = f"{SEQ_START}\n{USER}\n{prompt_text}\n{ASSISTANT}\n"

    # tokjenize prompt
    input_ids = tokenizer.encode(formatted_prompt, allowed_special="all")
    x = torch.tensor([input_ids], dtype=torch.long).to(device)

    print(f"User: {prompt_text}")
    print("Assistant: ", end="", flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            # catch the full tuple output
            outputs = model(x)

            # extract the actual tensor (usually the first item at index 0)
            # If your model uses Hugging Face classes, this might need to be outputs.logits
            logits_tensor = outputs[0]

            # now we can safely slice the tensor!
            next_token_logits = logits_tensor[0, -1, :]

            next_token_id = torch.argmax(next_token_logits).item()

            word = tokenizer.decode([next_token_id])
            print(word, end="", flush=True)

            next_token_tensor = torch.tensor([[next_token_id]], device=device)
            x = torch.cat((x, next_token_tensor), dim=1)

    print("\n")

In [ ]:
# test it!
generate_response("Explain the science behind blackholes", max_new_tokens=50)

User: Explain the science behind blackholes
Assistant: I do not have a physical body or a physical body. However, it is possible to find a way to explore the world and explore the world. 

1. Research the population: research the population of blackholes is a significant factor



In [ ]:
import torch
import torch.nn.functional as F

model.eval()

# NOTE: Change this to whatever token your model uses to end conversations!
# It might be "<|end|>", "<|eos|>", or "<|endoftext|>" depending on your tokenizer setup.
try:
    END_TOKEN = tokenizer.encode("<|endoftext|>", allowed_special="all")[0]
except:
    END_TOKEN = None # Fallback if you don't have one

def generate_response(prompt_text, max_new_tokens=150, temperature=0.7, top_k=40, device="cuda"):
    formatted_prompt = f"{SEQ_START}\n{USER}\n{prompt_text}\n{ASSISTANT}\n"

    input_ids = tokenizer.encode(formatted_prompt, allowed_special="all")
    x = torch.tensor([input_ids], dtype=torch.long).to(device)

    print(f"User: {prompt_text}")
    print("Assistant: ", end="", flush=True)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            outputs = model(x)
            logits = outputs[0][0, -1, :] # Grab the raw scores

            # 1. Apply Temperature (higher = more creative/random, lower = more focused)
            logits = logits / temperature

            # 2. Apply Top-K (keep only the top 40 most likely words, delete the rest)
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[-1]] = -float('Inf')

            # 3. Convert scores to probabilities and roll the dice!
            probs = F.softmax(logits, dim=-1)
            next_token_id = torch.multinomial(probs, num_samples=1).item()

            # 4. THE STOP SIGN: If the model outputs the End Token, break the loop!
            if END_TOKEN is not None and next_token_id == END_TOKEN:
                break

            word = tokenizer.decode([next_token_id])
            print(word, end="", flush=True)

            next_token_tensor = torch.tensor([[next_token_id]], device=device)
            x = torch.cat((x, next_token_tensor), dim=1)

    print("\n")

# Try it out! You can tweak temperature to make it more or less creative.
generate_response("Name the planets in our solar system", temperature=0.1)

NameError: name 'model' is not defined

#IDK

In [ ]:
# wandb eval
# This script needs these libraries to be installed:
#   numpy, transformers, datasets
import wandb

import os
import numpy as np
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": np.mean(predictions == labels)}


# download prepare the data
dataset = load_dataset("yelp_review_full")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(300))

small_train_dataset = small_train_dataset.map(tokenize_function, batched=True)
small_eval_dataset = small_train_dataset.map(tokenize_function, batched=True)

# download the model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)

# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="nanobot-1"

# save your trained model checkpoint to wandb
os.environ["WANDB_LOG_MODEL"]="true"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="false"

# pass "wandb" to the 'report_to' parameter to turn on wandb logging
training_args = TrainingArguments(
    output_dir='models',
    report_to="wandb",
    logging_steps=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    eval_steps=20,
    max_steps = 100,
    save_steps = 100
)

# define the trainer and start training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

# [optional] finish the wandb run, necessary in notebooks
wandb.finish()

In [ ]:
# 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

import os
import numpy as np
import wandb
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification

# model that accepts 4096 tokens
from transformers import AutoModelForSequenceClassification

# 1. Assume `tokenizer` is your already trained rustbpe.Tokenizer
# tokenizer = Tokenizer()
# tokenizer.train_from_iterator(...)

# 2. Instantiate the wrapper
hf_tokenizer = HFCompatibleTokenizer(tokenizer, max_length=SEQ_LEN, pad_token_id=0)

def tokenize_function(examples):
    # Process through our custom wrapper
    return hf_tokenizer(examples["text"], padding="max_length", truncation=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": np.mean(predictions == labels)}

# Download prepare the data
dataset = load_dataset("yelp_review_full")

small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(300))

# Map using the custom tokenize_function
small_train_dataset = small_train_dataset.map(tokenize_function, batched=True)
small_eval_dataset = small_eval_dataset.map(tokenize_function, batched=True) # Fixed this line

# Download the model
model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/longformer-base-4096",
    num_labels=5
)


model.resize_token_embeddings(VOCAB_SIZE) # Match your VOCAB_SIZE
model.config.pad_token_id = hf_tokenizer.pad_token_id # Let the model know which ID is padding

# Set the wandb variables
os.environ["WANDB_PROJECT"] = "nanobot-1"
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_WATCH"] = "false"

training_args = TrainingArguments(
    output_dir='models',
    report_to="wandb",
    logging_steps=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    eval_steps=20,
    max_steps=100,
    save_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# Finish the wandb run
wandb.finish()